# Full Fine-Tuning vs. LoRA (PEFT) for Dialogue Summarization

Trains **FLAN-T5-base** two ways on [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) —
full fine-tuning (every parameter updated) vs. a **LoRA adapter** (base
frozen) — and compares both against the untouched zero-shot model on ROUGE,
training time, and trainable-parameter percentage. Also runs a LoRA rank
sweep (r = 4/16/32).

Reuses the `src/` package from the companion GitHub repo.

> **Kaggle setup:** this notebook trains real models, so you need a GPU.
> Under *Notebook Settings*, set **Accelerator** to a GPU (T4 x2 or P100) and
> turn **Internet** ON (needed to clone the repo and download the
> model/dataset from Hugging Face).

In [ ]:
# Install pinned dependencies
!pip install -q transformers==4.40.0 datasets==2.19.0 accelerate==0.29.3 peft==0.10.0 rouge-score==0.1.2

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

If the cell above prints `CUDA available: False`, stop and enable a GPU
accelerator in Notebook Settings before continuing — full fine-tuning on CPU
will be extremely slow.

In [ ]:
# Clone the project repo and make src/ importable.
# Replace the URL below if you rename or fork the repo.
import sys, subprocess, pathlib

REPO_URL = "https://github.com/shimaaelbana/dialogue-summarization-finetune-peft.git"
REPO_DIR = pathlib.Path("dialogue-summarization-finetune-peft")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)

sys.path.append(str(REPO_DIR.resolve()))

## 1. Zero-shot baseline vs. full fine-tune vs. LoRA fine-tune

In [ ]:
from src.pipeline import run_comparison

# Kept small so a full run finishes in a reasonable time on a free-tier GPU.
# Increase train_size / num_train_epochs for a more thorough (slower) run.
comparison_df = run_comparison(
    model_name="google/flan-t5-base",
    train_size=1000,
    eval_size=100,
    test_size=50,
    num_train_epochs=1,
    lora_r=32,
)
comparison_df

In [ ]:
import os

os.makedirs("results", exist_ok=True)
comparison_df.to_csv("results/comparison.csv", index=False)
print("Saved results/comparison.csv")

### Visualize: ROUGE-L vs. trainable parameter percentage

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(comparison_df["model"], comparison_df["rougeL"])
axes[0].set_ylabel("ROUGE-L (F-measure)")
axes[0].set_title("Summarization quality")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(comparison_df["model"], comparison_df["trainable_pct"])
axes[1].set_ylabel("Trainable parameters (%)")
axes[1].set_title("Training cost")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150)
plt.show()

## 2. LoRA rank sweep

Original extension beyond the source lab: how does the LoRA rank `r` trade
off parameter count against summarization quality?

In [ ]:
from src.pipeline import run_lora_rank_sweep

sweep_df = run_lora_rank_sweep(
    model_name="google/flan-t5-base",
    ranks=[4, 16, 32],
    train_size=1000,
    eval_size=100,
    test_size=50,
    num_train_epochs=1,
)
sweep_df

In [ ]:
sweep_df.to_csv("results/lora_rank_sweep.csv", index=False)
print("Saved results/lora_rank_sweep.csv")

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))

ax1.plot(sweep_df["lora_r"], sweep_df["rougeL"], marker="o", color="tab:blue", label="ROUGE-L")
ax1.set_xlabel("LoRA rank (r)")
ax1.set_ylabel("ROUGE-L (F-measure)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(sweep_df["lora_r"], sweep_df["trainable_pct"], marker="s", color="tab:red", label="Trainable %")
ax2.set_ylabel("Trainable parameters (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("LoRA rank vs. quality and parameter cost")
plt.tight_layout()
plt.savefig("results/lora_rank_sweep.png", dpi=150)
plt.show()

## Findings

_Fill this in after running the cells above:_

- How much ROUGE does full fine-tuning gain over zero-shot, and how much of
  that gain does LoRA (r=32) recover?
- Is the training-time gap between full fine-tuning and LoRA as large as the
  trainable-parameter gap suggests?
- In the rank sweep, does ROUGE-L keep improving from r=4 to r=32, or does it
  plateau? Where's the best quality-per-trainable-parameter tradeoff?